In [28]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt

import os

In [29]:
path = os.getcwd()
print(path)

/Users/macfadda/Git/xai-visualization_rules_fi/notebooks


# Loading a dataset and preparing it

In [30]:
datasets=['titanic_c.csv','german_credit.csv']

In [31]:
source_file = f'../datasets/{datasets[1]}'
class_field = 'default'
# Load and transform dataset
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [32]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

In [33]:
df_v = data_to_plot(feature_names=feature_names, real_feature_names=real_feature_names, instance_number=3, x_train=X_train, rules=rules, feature_importance='shap')
df_v

,type,name,rname,min,max,q1,median,q3,mean,std,feature_importance,category,count,inst,op,thr,is_continuous,thr2
32,categorical,present_emp_since=... < 1 year,present_emp_since,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.060481,... < 1 year,120.0,0,NaN,NaN,NaN,NaN
10,categorical,account_check_status=no checking account,account_check_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.059744,no checking account,276.0,0,NaN,NaN,NaN,NaN
12,categorical,credit_history=critical account/ other credits...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.044226,critical account/ other credits existing (not ...,205.0,0,<=,0.908596,True,NaN
26,categorical,savings=.. >= 1000 DM,savings,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.037394,.. >= 1000 DM,37.0,0,<=,0.717686,True,NaN
48,categorical,other_installment_plans=none,other_installment_plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.026069,none,563.0,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24,categorical,purpose=repairs,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000091,repairs,10.0,0,NaN,NaN,NaN,NaN
50,categorical,housing=for free,housing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000056,for free,75.0,0,NaN,NaN,NaN,NaN
14,categorical,credit_history=existing credits paid back duly...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000053,existing credits paid back duly till now,362.0,1,NaN,NaN,NaN,NaN
22,categorical,purpose=furniture/equipment,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,furniture/equipment,6.0,0,<=,0.183708,True,NaN


## Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [34]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [35]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [36]:
inst = X_train.iloc[128].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [  27 4526    4    2   32    2    2    0    0    0    1    0    1    0
    0    0    0    0    0    0    0    0    0    1    0    0    1    0
    0    0    0    0    1    0    0    0    0    0    0    1    0    0
    1    0    0    1    0    0    0    1    0    1    0    0    0    0
    1    0    1    0    1]
True class  0
Predicted class  [0]


In [37]:
real_inst = inst
real_inst

array([  27, 4526,    4,    2,   32,    2,    2,    0,    0,    0,    1,
          0,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    1,    0,    0,    1,    0,    0,    0,    0,    0,    1,
          0,    0,    0,    0,    0,    0,    1,    0,    0,    1,    0,
          0,    1,    0,    0,    0,    1,    0,    1,    0,    0,    0,
          0,    1,    0,    1,    0,    1])

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [38]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:].values}
explainer.fit(config)

In [39]:
exp = explainer.explain(inst)

In [40]:
shap_feature_importance=exp.exp

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.

In [41]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [42]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 2.27797454  3.35504085  0.94540357  1.07634233  0.04854891 -0.72456474
 -0.43411405  1.65027399 -0.61477862 -0.25898489 -0.80681063  4.17385345
 -0.6435382  -0.32533856 -1.03489416 -0.20412415 -0.22941573 -0.33068147
  1.75885396 -0.34899122 -0.60155441 -0.15294382 -0.09298136 -0.46852129
 -0.12038585 -0.08481889 -0.23623492 -1.21387736 -0.36174054 -0.24943031
  2.15526362 -0.59715086 -0.45485883 -0.73610476 -0.43875307  4.23307441
 -0.65242771 -0.23958675 -0.32533856  0.90192655  4.72581563 -0.2259448
 -3.15238005 -0.54212562 -0.70181003 -0.63024248  2.30354212 -0.40586384
  0.49329429 -0.23958675  2.88675135 -1.59227935 -0.46170508  2.46388049
 -1.33747696 -0.13206764 -0.5        -1.21387736  1.21387736 -0.20412415
  0.20412415]
Predicted class  [1]


## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.

## LIME tabular explainer

In [43]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)

In [44]:
lime_feature_importance=lime_exp.exp.as_list()
lime_feature_importance[1:3]

[('credit_history=all credits at this bank paid back duly',
  -9.80729285586033e-10),
 ('present_emp_since=unemployed', -8.905493791597656e-10)]

In [45]:
#lime_exp.plot_features_importance()

### LORE explainer

In [46]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [47]:
rules =exp.expDict['rule']['premise']

## Prepare data to plot

In [48]:
def data_to_plot(
        feature_names=feature_names, real_feature_names=real_feature_names,
        instance_number=None, x_train=None, rules=None,
        feature_importance=None):
    feature_list =[]
    #Convert the list of tuples generated by lime in a dict
    if feature_importance is 'lime':
        lime_dict = {}
        for (key, value) in lime_feature_importance:
            lime_dict.setdefault(key, value)
    for i, el in enumerate(feature_names):
        f ={}
        if el in numeric_columns:
            f['type'] = 'numeric'
            f['name'] = el
            f['rname'] = real_feature_names[i]
            if x_train is not None:
                f['min'] = x_train[el].min()
                f['max'] = x_train[el].max()
                f['q1'] = x_train[el].quantile(0.25)
                f['median'] = x_train[el].quantile(0.50)
                f['q3'] = x_train[el].quantile(0.75)
                f['mean'] = x_train[el].mean()
                f['std'] = x_train[el].std()
        else:
            f['type'] = 'categorical'
            f['name'] = el
            f['rname'] = el.split('=')[0]
            f['category'] = el.split('=',1)[1]
            if x_train is not None:
                f['count'] = x_train[el].sum()

        if feature_importance is 'lime':
            f['feature_importance'] = lime_dict[el]
        if feature_importance is 'shap':
            f['feature_importance'] = shap_feature_importance[1][i]
        feature_list.append(f)
    df =pd.DataFrame.from_records(feature_list)

    if instance_number:
        inst = X_train.iloc[instance_number].values
        df['inst'] = inst
    if rules is not None:
        df_rules = pd.DataFrame.from_records(rules)
        df = df.merge(df_rules,how='left',left_on='name',right_on='att')
        df = df.drop('att', axis=1)
        thr2_list=[]
        for i, row in df.iterrows():
            if row['op']== '>' or row['op']== '>=':
                thr2_list.append(row['max'])
                continue
            if row['op']== '<' or row['op']== '<=':
                thr2_list.append(row['min'])
                continue
            else:
                thr2_list.append(np.nan)
        df['thr2'] = thr2_list
    df.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return df

# Plotting functions

In [49]:
def single_feature_importance_plot(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).mark_bar(
    ).encode(
        x=alt.X(
            field='feature_importance',
            type='quantitative',
            title=None
        ),
        y=alt.Y(
            field='name',
            type='nominal',
            title=None,
            axis=None
        ),
        color=alt.condition('datum.feature_importance > 0', alt.value('#285588'), alt.value('#E36273')),
        tooltip=[alt.Tooltip(field="name"),alt.Tooltip(field="feature_importance")]
    )
    return chart.properties(
        height=20,
        width=100
    )

In [65]:
def single_rule_plot_numeric(dataframe,rw):
    data = dataframe[dataframe['name'] == rw['name']]
    p=alt.Chart(
        data
    ).mark_point(
        color='black' if rw['is_continuous'] == True else 'black',
        size=70,
        shape='diamond',
        filled=True
    ).encode(
        x=alt.X(
            field='inst',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        tooltip=[alt.Tooltip(field='inst', title=rw['name'])]
    )

    t_min = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=-10,
        align='right'
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None
        ),
        text='min:N'
    )

    t_max = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=10,
        align='left'
    ).encode(
        x=alt.X(
            field='max',
            type='quantitative',
            title=None
        ),
        text='max:N'
    )

    

    b =alt.Chart(
        data
    ).mark_bar(
        color='#fcc40f',size=5,
        stroke='white'
    ).encode(
        x=alt.X(
            field='thr',
            type='quantitative',
            title=None,
        ),
        x2='thr2',
        y=alt.Y(
            field='name',
            type='nominal',
            title=None
        ),

    )



    l =alt.Chart(
        data
    ).mark_bar(
        color='grey',size=1
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2='max',
        y=alt.Y(field='name',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
    )


    q1_m = alt.Chart(
        data
    ).mark_bar(
        stroke='white',
        color='lightgrey',
        size=18
    ).encode(
        x=alt.X(
            field='q1',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='median'
        ),
    )

    m_q3 = alt.Chart(
        data
    ).mark_bar(
        stroke='white',
        color='lightgrey',
        size=18,
    ).encode(
        x=alt.X(
            field='median',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='q3'
        )
    )
    
    return alt.layer(l,q1_m,m_q3,b,p).properties(
        height=20,
        width=300        
    )

In [67]:
def single_index_text(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).transform_calculate(
        label ="datum.type=='categorical' ? datum.name : datum.name +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
    ).mark_text(
        color='black',
        align='left',
        dx=-50,
        fontSize=13
    ).encode(
            text=alt.Text(
            field='label',
            type='nominal',
            title=None
        )
    )
    return chart.properties(
        height=20,
        width=101
    )

In [66]:
def plot_rules(dataframe, only_rules=False):
    ti_list=[]
    rp_list=[]
    fi_list=[]
    
    for i, row in dataframe.iterrows():
        if row['inst']!=0:
            if ((only_rules == True) and (row['is_continuous']!= True)):
                pass
            else:
                sti = single_index_text(dataframe, row)
                if row['type']== 'numeric':
                    srp = single_rule_plot_numeric(dataframe, row)
                else:
                    srp = single_rule_plot_qualit(dataframe, row)
                sfi = single_feature_importance_plot(dataframe, row)
                ti_list.append(sti)
                rp_list.append(srp)
                fi_list.append(sfi)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI').resolve_scale(
    x='shared'
)
    final_chart = alt.hconcat(
        fi_concat, rp_concat, ti_concat
    )

    return final_chart.configure(
       # background='#F5F5F5',
        padding=20
    ).configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [91]:
def single_rule_plot_qualit(dataframe, rw):
    name= rw['name'].split('=')[0]
    data = dataframe[dataframe['rname'] == name]
    
    base= alt.Chart(
        data
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'descending')]
    ).transform_calculate(
        midStack='(datum.count_start+datum.count_end)/2'
    )
    
    
    bar = base.mark_bar(
        stroke='white',
        color='lightgrey'
    ).encode(
        x='count_start:Q',
        x2='count_end:Q',
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    bar_r = base.mark_bar(
         stroke='#fcc40f'
     ).encode(
        x='count_start:Q',
        x2='count_end:Q',
        color=alt.condition('datum.is_continuous && datum.inst==1',alt.value("#fcc40f"),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.0001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )

    r=base.mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='name:N',
        color=alt.condition('datum.is_continuous',alt.value('#fcc40f'),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    dot =base.mark_point(
        size=70,
        shape='diamond',
        color='black',
        filled=True
    ).encode(
        x=alt.X(
            field='midStack',
            type='quantitative',
            title=None,
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
    )
    

    return alt.layer(bar,bar_r,dot).properties(
        height=20,
        width=300,
    )

# Plot

In [92]:
plot_rules(df_v, only_rules=False)

alt.HConcatChart(...)

In [93]:
plot_rules(df_v, only_rules=True)

alt.HConcatChart(...)

In [58]:
df_v[df_v['rname']=='credit_history']

,type,name,rname,min,max,q1,median,q3,mean,std,feature_importance,category,count,inst,op,thr,is_continuous,thr2
12,categorical,credit_history=critical account/ other credits...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.044226,critical account/ other credits existing (not ...,205.0,0,<=,0.908596,True,NaN
11,categorical,credit_history=all credits at this bank paid b...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.006010,all credits at this bank paid back duly,38.0,0,NaN,NaN,NaN,NaN
15,categorical,credit_history=no credits taken/ all credits p...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.003352,no credits taken/ all credits paid back duly,28.0,0,NaN,NaN,NaN,NaN
13,categorical,credit_history=delay in paying off in the past,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000823,delay in paying off in the past,67.0,0,NaN,NaN,NaN,NaN
14,categorical,credit_history=existing credits paid back duly...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000053,existing credits paid back duly till now,362.0,1,NaN,NaN,NaN,NaN


# label encoder scikit